# Arctic Grid Spectral Analysis — CIMR Sea Ice Concentration

This notebook performs spectral analysis of **Sea Ice Concentration (SIC)** predictions from CIMR-like data at two resolutions:
- **x50** — 25 km resolution (full period, Feb 1–15)
- **x10** — 5 km resolution (Feb 8–15)

Both files contain:
- `pred_SIC`: 4DVarNet reconstruction/forecast
- `models_SIC`: model reference (used as ground truth)

The **last 3 time steps** are forecast lead times (lead-0, lead-1, lead-2).  
Spectral evaluation is performed:
1. Over the **analysis period** (non-forecast days) for each resolution.
2. Per **lead time** (0, 1, 2), comparing: Reference (`models_SIC`), Persistence (last analysis day of `models_SIC`), and Forecast (`pred_SIC`), for both resolutions.


In [ ]:
import numpy as np
import xarray as xr
import scipy.signal
from skimage.draw import line
import matplotlib.pyplot as plt
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
import os, glob, re
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import numpy as np

# ── Data locations ────────────────────────────────────────────────────────────
BASE        = '/dmidata/users/maxb/4dvarnet-starter/NetCDF_tests'
CIMR_DIR    = '/dmidata/users/maxb/CROSCIM_dataset/out_CIMR'      # passive-microwave SIC obs (dense)
ASIP_DIR    = '/dmidata/users/maxb/CROSCIM_dataset/out_ASIP'      # high-resolution SIC obs

N_FORECAST  = 3   # last 3 time steps of each file are forecast lead-0 / 1 / 2

# ── The 4 benchmark methods (file prefix + resolutions produced) ──────────────
# SIC port: multires methods produce 3 resolutions (x50/x10/x2); the single-resolution
# baselines use the finest grid (x2). Adjust prefixes/files for the actual SIC runs.
METHODS = {
    'UNet-UOAI':           {'prefix': 'UNet_UOAI',           'res': ['x50', 'x10', 'x2']},
    'UNet-UOAI-res2':      {'prefix': 'UNet_UOAI_res2',      'res': ['x2']},
    'UNet-Unrolling':      {'prefix': 'UNet_unrolling',      'res': ['x50', 'x10', 'x2']},
    'UNet-Unrolling-res2': {'prefix': 'UNet_unrolling_res2', 'res': ['x2']},
}
# Ground-truth variable depends on resolution: model-based at x50/x10, but the
# x2 (1 km) target is OBSERVATION-based (tgt_SIC, not models_SIC) and contains NaN.
GT_VAR = {'x50': 'models_SIC', 'x10': 'models_SIC', 'x2': 'tgt_SIC'}

# Method whose single-sequence maps + per-region spectral plot we display.
PLOT_METHOD = 'UNet-Unrolling'

# ── Discover the 25 sequences from the file names (idx, start, end) ───────────
_pat = re.compile(r'_seq(\d{2})_(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})_x10\.nc$')
SEQUENCES = []
for f in sorted(glob.glob(f'{BASE}/{METHODS[PLOT_METHOD]["prefix"]}_seq*_x10.nc')):
    m = _pat.search(os.path.basename(f))
    if m:
        SEQUENCES.append((int(m.group(1)), m.group(2), m.group(3)))
SEQUENCES.sort()
print(f'Discovered {len(SEQUENCES)} sequences ({SEQUENCES[0][1]} … {SEQUENCES[-1][2]})')

# ── Optionally keep ONLY in-distribution sequences: window within the model
#    TRAINING period (read from the inference config). Set False to use all 25.
FILTER_INDIST = True
import yaml, datetime as _dt
TRAIN_CONFIG = '/dmidata/users/maxb/4dvarnet-starter/config/xp/CROSCIM/UNet_solvers/base_arctic_croscim_test_sic_UOAI_supervised_forecast.yaml'
with open(TRAIN_CONFIG) as _f:
    _ts, _te = yaml.safe_load(_f)['datamodule']['domains']['train']['time']['_args_']
TRAIN_START, TRAIN_END = _dt.date.fromisoformat(_ts), _dt.date.fromisoformat(_te)
print(f'Training period (from config): {TRAIN_START} → {TRAIN_END}')
if FILTER_INDIST:
    _all_seqs = SEQUENCES
    SEQUENCES = [(i, s, e) for (i, s, e) in _all_seqs
                 if _dt.date.fromisoformat(s) >= TRAIN_START and _dt.date.fromisoformat(e) <= TRAIN_END]
    print(f'FILTER_INDIST=True → kept {len(SEQUENCES)}/{len(_all_seqs)} in-distribution: {[i for i,_,_ in SEQUENCES]}')
else:
    print(f'FILTER_INDIST=False → using all {len(SEQUENCES)} sequences (incl. OOD winter)')

SEQ01 = SEQUENCES[0]            # first in-distribution sequence — used for the example maps
print(f'First sequence (maps): seq{SEQ01[0]:02d}  {SEQ01[1]} → {SEQ01[2]}')


def method_file(method, idx, res):
    """Path of the NetCDF for a given method / sequence index / resolution ('x50'|'x10'), or None."""
    hits = glob.glob(f'{BASE}/{METHODS[method]["prefix"]}_seq{idx:02d}_*_{res}.nc')
    return hits[0] if hits else None


# Forecast dates of the first sequence (last N_FORECAST days) — for the obs panel.
import xarray as xr
with xr.open_dataset(method_file(PLOT_METHOD, SEQ01[0], 'x10')) as _ds:
    _t = [str(t)[:10] for t in _ds.time.values]
FORECAST_DATES = _t[-N_FORECAST:]     # lead-0, lead-1, lead-2 of seq01
print(f'Forecast dates seq01 (lead-0/1/2): {FORECAST_DATES}')


# ── Native grid CRS (EPSG:3411 / NSIDC North polar stereo, lat_ts=70, lon_0=-45) ─
# Plotting pcolormesh in this native projection (xc/yc in metres) gives clean,
# artefact-free maps — no antimeridian/pole streaks from curvilinear lon/lat.
import cartopy.crs as ccrs, cartopy.feature as cfeature
DATA_CRS = ccrs.NorthPolarStereo(
    central_longitude=-45, true_scale_latitude=70,
    globe=ccrs.Globe(semimajor_axis=6378273.0, semiminor_axis=6356889.449))

def polar_map(ax, xc, yc, data, cmap='viridis', vmin=None, vmax=None, extent=None):
    """Artefact-free pcolormesh on the native polar-stereographic grid."""
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax.coastlines(linewidth=0.5, zorder=3)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    pcm = ax.pcolormesh(xc, yc, data, transform=DATA_CRS, cmap=cmap,
                        vmin=vmin, vmax=vmax, shading='auto', zorder=0)
    return pcm

# Sub-regions [lon_min, lon_max, lat_min, lat_max] and colours (shared by all map cells)
REGIONS = {
    'Barents Sea':      [15,  70, 70, 82],
    'Kara Sea':         [60, 100, 70, 82],
    'Greenland / Fram': [-20, 20, 70, 85],
}
REGION_COLORS = {'Barents Sea': '#d62728', 'Kara Sea': '#1f77b4', 'Greenland / Fram': '#8B2FC9'}  # canonical (Kara = nice blue)

# Generic polar projection + circular boundary (used by the segment-visualisation cell)
from matplotlib.path import Path as _MplPath
proj = ccrs.NorthPolarStereo()
_th0 = np.linspace(0, 2 * np.pi, 200)
circle = _MplPath(np.vstack([np.sin(_th0) * 0.5 + 0.5, np.cos(_th0) * 0.5 + 0.5]).T)


In [ ]:
# ── Sea Ice Concentration observations vs ground truth — CIMR | ASIP | GT(model) ─
# Native polar-stereographic grid (xc/yc) → clean, artefact-free maps.
import warnings; warnings.filterwarnings('ignore')
import numpy as np, xarray as xr, matplotlib.pyplot as plt

date = FORECAST_DATES[-1]                       # lead-2 of seq01

ds_ci = xr.open_dataset(f'{CIMR_DIR}/CIMR5km_{date}.nc')
ds_as = xr.open_dataset(f'{ASIP_DIR}/ASIP_{date}.nc')
ds_gt = xr.open_dataset(method_file(PLOT_METHOD, SEQ01[0], 'x10'))

panels = [
    ('CIMR',       ds_ci['SIC'].squeeze().values, ds_ci.xc.values, ds_ci.yc.values),
    ('ASIP',       ds_as['SIC'].squeeze().values, ds_as.xc.values, ds_as.yc.values),
    ('GT (model)', ds_gt['models_SIC'].sel(time=date).values, ds_gt.xc.values, ds_gt.yc.values),
]
VMIN, VMAX = 0.0, 1.0

fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.4), subplot_kw={'projection': DATA_CRS})
pcm = None
for ax, (title, sic, xc, yc) in zip(axes, panels):
    pcm = polar_map(ax, xc, yc, sic, cmap='viridis', vmin=VMIN, vmax=VMAX)
    ax.set_title(title, fontsize=13, fontweight='bold')
for d in (ds_ci, ds_as, ds_gt):
    d.close()

cax = fig.add_axes([0.30, 0.10, 0.40, 0.035])
fig.colorbar(pcm, cax=cax, orientation='horizontal').set_label('Sea Ice Concentration', fontsize=11)

fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.20, wspace=0.04)
fig.suptitle(f'Sea Ice Concentration observations vs ground truth — {date}', fontsize=14, y=0.99)
plt.show()


In [ ]:
# ── Load the FIRST sequence (seq01) for the example maps ─────────────────────
# One method (PLOT_METHOD), all three resolutions; downstream map cells use cristal_data.
import numpy as np
import xarray as xr
import os

print("=" * 80)
print(f"LOADING seq{SEQ01[0]:02d} ({SEQ01[1]} → {SEQ01[2]}) — method {PLOT_METHOD}")
print("=" * 80)

cristal_data = {}   # keyed by 'x50', 'x10' and 'x2'

for key in ['x50', 'x10', 'x2']:
    fpath = method_file(PLOT_METHOD, SEQ01[0], key)
    print(f"\n{'─'*60}\nLoading {key} — {os.path.basename(fpath)}")
    ds = xr.open_dataset(fpath)

    xc = ds.xc.values
    yc = ds.yc.values
    dx = abs(np.mean(np.diff(xc))) / 1000.0
    dy = abs(np.mean(np.diff(yc))) / 1000.0

    nt = ds.sizes['time']
    n_analysis = nt - N_FORECAST

    pred_all   = ds['pred_SIC'].values
    models_all = ds[GT_VAR[key]].values
    lat_2d     = ds['lat'].values
    lon_2d     = (ds['lon'].values + 180) % 360 - 180
    times      = ds.time.values

    cristal_data[key] = {
        'xc': xc, 'yc': yc, 'dx': dx, 'dy': dy, 'resolution_km': dx,
        'times_all': times,
        'times_analysis': times[:n_analysis],
        'times_forecast': times[n_analysis:],
        'pred_analysis':   pred_all[:n_analysis],
        'models_analysis': models_all[:n_analysis],
        'pred_forecast':   pred_all[n_analysis:],
        'models_forecast': models_all[n_analysis:],
        'persistence':     models_all[[n_analysis - 1] * N_FORECAST],
        'lat': lat_2d, 'lon': lon_2d,
    }
    print(f"  Grid {len(xc)}×{len(yc)}, dx={dx:.1f} km | {nt} days "
          f"({n_analysis} analysis + {N_FORECAST} forecast)")
    ds.close()

print(f"\n{'='*80}\n✓ seq01 loaded ({PLOT_METHOD})\n{'='*80}")


## Geographic Maps — Reference | Persistence | Prediction  (per resolution + regional zooms)

For each resolution (x50, x10) a figure is produced with:
- **Row 0** — global Arctic view (North Polar Stereo)
- **Rows 1–3** — zoomed-in views for three sea-ice regions of interest
- **Columns** — Reference (`models_SIC`), Persistence (last analysis day of `models_SIC`), Prediction (`pred_SIC`)

Displayed at forecast lead `MAP_LEAD` (0/1/2).

In [ ]:
# ── Geographic maps: Reference | ML Prediction — native grid (artefact-free) ──
import time, numpy as np, matplotlib.pyplot as plt, cartopy.crs as ccrs

MAP_LEAD = 2     # lead time shown for the global/zoom maps

def draw_box(ax, ext, color, label=None, lw=2):
    npts = 50
    lo = np.concatenate([np.linspace(ext[0],ext[1],npts), np.full(npts,ext[1]),
                         np.linspace(ext[1],ext[0],npts), np.full(npts,ext[0])])
    la = np.concatenate([np.full(npts,ext[2]), np.linspace(ext[2],ext[3],npts),
                         np.full(npts,ext[3]), np.linspace(ext[3],ext[2],npts)])
    ax.plot(lo, la, color=color, lw=lw, transform=ccrs.PlateCarree(), zorder=10)

col_keys, col_titles = ['models_forecast', 'pred_forecast'], ['Reference', 'ML Prediction']

t0 = time.time()
for key in ['x50', 'x10', 'x2']:
    d = cristal_data[key]; xc, yc = d['xc'], d['yc']
    fields = [d['models_forecast'][MAP_LEAD], d['pred_forecast'][MAP_LEAD]]
    allv = np.concatenate([f.ravel() for f in fields])
    vmin, vmax = np.nanpercentile(allv, 1), np.nanpercentile(allv, 99)
    fc_date = d['times_forecast'][MAP_LEAD].astype('datetime64[D]')

    fig, axes = plt.subplots(4, 2, figsize=(10, 18), subplot_kw={'projection': DATA_CRS})
    pcm = None
    for j, arr in enumerate(fields):
        pcm = polar_map(axes[0, j], xc, yc, arr, vmin=vmin, vmax=vmax)
        for rname, ext in REGIONS.items():
            draw_box(axes[0, j], ext, REGION_COLORS[rname])
        axes[0, j].set_title(col_titles[j], fontsize=12, fontweight='bold')
    for i, (rname, ext) in enumerate(REGIONS.items(), start=1):
        for j, arr in enumerate(fields):
            polar_map(axes[i, j], xc, yc, arr, vmin=vmin, vmax=vmax, extent=ext)
        axes[i, 0].text(0.02, 0.95, rname, color=REGION_COLORS[rname], fontsize=11,
                        fontweight='bold', transform=axes[i, 0].transAxes,
                        bbox=dict(facecolor='white', alpha=0.75, edgecolor=REGION_COLORS[rname], lw=1.5))

    cax = fig.add_axes([0.25, 0.05, 0.5, 0.012])
    fig.colorbar(pcm, cax=cax, orientation='horizontal').set_label('Sea Ice Concentration', fontsize=11)
    fig.suptitle(f'Sea Ice Concentration — {key} ({d["resolution_km"]:.0f} km)  |  Lead-{MAP_LEAD} ({fc_date})',
                 fontsize=13, y=0.93)
    fig.subplots_adjust(hspace=0.10, wspace=0.04, top=0.90, bottom=0.07)
    plt.show()
print(f"Maps done in {time.time()-t0:.1f}s")


In [ ]:
# ── Error maps: Reference | ML Prediction | Δ — native grid (artefact-free) ──
import time as _t, numpy as np, matplotlib.pyplot as plt

t0 = _t.time()
for key in ['x50', 'x10', 'x2']:
    d = cristal_data[key]; xc, yc = d['xc'], d['yc']
    for lt in range(N_FORECAST):
        ref, pred = d['models_forecast'][lt], d['pred_forecast'][lt]
        diff = ref - pred
        allv = np.concatenate([ref.ravel(), pred.ravel()])
        vmin, vmax = np.nanpercentile(allv, 1), np.nanpercentile(allv, 99)
        amax = max(np.nanpercentile(np.abs(diff), 99), 0.1)
        fc_date = d['times_forecast'][lt].astype('datetime64[D]')
        panels = [(ref, 'viridis', vmin, vmax, 'Reference'),
                  (pred, 'viridis', vmin, vmax, 'ML Prediction'),
                  (diff, 'coolwarm', -amax, amax, 'Δ = Reference − Prediction')]

        fig, axes = plt.subplots(4, 3, figsize=(14, 18), subplot_kw={'projection': DATA_CRS})
        ims = [None, None, None]
        for j, (arr, cmap, vmn, vmx, title) in enumerate(panels):
            ims[j] = polar_map(axes[0, j], xc, yc, arr, cmap=cmap, vmin=vmn, vmax=vmx)
            for rname, ext in REGIONS.items():
                draw_box(axes[0, j], ext, REGION_COLORS[rname])
            axes[0, j].set_title(title, fontsize=12, fontweight='bold')
        for i, (rname, ext) in enumerate(REGIONS.items(), start=1):
            for j, (arr, cmap, vmn, vmx, _) in enumerate(panels):
                polar_map(axes[i, j], xc, yc, arr, cmap=cmap, vmin=vmn, vmax=vmx, extent=ext)
            axes[i, 0].text(0.02, 0.95, rname, color=REGION_COLORS[rname], fontsize=11,
                            fontweight='bold', transform=axes[i, 0].transAxes,
                            bbox=dict(facecolor='white', alpha=0.75, edgecolor=REGION_COLORS[rname], lw=1.5))

        cax1 = fig.add_axes([0.08, 0.05, 0.35, 0.012])
        cax2 = fig.add_axes([0.575, 0.05, 0.35, 0.012])
        fig.colorbar(ims[0], cax=cax1, orientation='horizontal').set_label('Sea Ice Concentration', fontsize=11)
        fig.colorbar(ims[2], cax=cax2, orientation='horizontal').set_label('Δ Sea Ice Concentration', fontsize=11)
        rmse = float(np.sqrt(np.nanmean(diff**2))); bias = float(np.nanmean(diff))
        fig.suptitle(f'Sea Ice Concentration {key} ({d["resolution_km"]:.0f} km) | Lead-{lt} ({fc_date})  '
                     f'—  RMSE={rmse:.4f},  bias={bias:+.4f}', fontsize=13, y=0.93)
        fig.subplots_adjust(hspace=0.10, wspace=0.04, top=0.90, bottom=0.07)
        plt.show()
print(f"Error maps done in {_t.time()-t0:.1f}s")


In [ ]:
# ── Anomaly comparison: True | ML forecast — native grid (artefact-free) ──────
# Anomalies relative to the ML reconstruction on the last analysis day.
import time as _t, numpy as np, matplotlib.pyplot as plt

t0 = _t.time()
for key in ['x50', 'x10', 'x2']:
    d = cristal_data[key]; xc, yc = d['xc'], d['yc']
    estim = d['pred_analysis'][-1]
    for lt in range(N_FORECAST):
        fc_date = d['times_forecast'][lt].astype('datetime64[D]')
        true_anom = d['models_forecast'][lt] - estim
        ml_anom   = d['pred_forecast'][lt]   - estim
        amax = max(max(np.nanpercentile(np.abs(a), 99) for a in (true_anom, ml_anom)), 0.05)
        panels = [(true_anom, 'True Anomaly (Reference − Estimate)'),
                  (ml_anom,   'ML Forecast Anomaly (Forecast − Estimate)')]

        fig, axes = plt.subplots(4, 2, figsize=(10, 18), subplot_kw={'projection': DATA_CRS})
        pcm = None
        for j, (arr, title) in enumerate(panels):
            pcm = polar_map(axes[0, j], xc, yc, arr, cmap='coolwarm', vmin=-amax, vmax=amax)
            for rname, ext in REGIONS.items():
                draw_box(axes[0, j], ext, REGION_COLORS[rname])
            axes[0, j].set_title(title, fontsize=12, fontweight='bold')
        for i, (rname, ext) in enumerate(REGIONS.items(), start=1):
            for j, (arr, _) in enumerate(panels):
                polar_map(axes[i, j], xc, yc, arr, cmap='coolwarm', vmin=-amax, vmax=amax, extent=ext)
            axes[i, 0].text(0.02, 0.95, rname, color=REGION_COLORS[rname], fontsize=11,
                            fontweight='bold', transform=axes[i, 0].transAxes,
                            bbox=dict(facecolor='white', alpha=0.75, edgecolor=REGION_COLORS[rname], lw=1.5))

        cax = fig.add_axes([0.25, 0.05, 0.5, 0.012])
        fig.colorbar(pcm, cax=cax, orientation='horizontal').set_label('Δ Sea Ice Concentration', fontsize=11)
        fig.suptitle(f'Sea Ice Concentration anomalies — {key} ({d["resolution_km"]:.0f} km) | Lead-{lt} ({fc_date})',
                     fontsize=13, y=0.93)
        fig.subplots_adjust(hspace=0.10, wspace=0.04, top=0.90, bottom=0.07)
        plt.show()
print(f"Anomaly maps done in {_t.time()-t0:.1f}s")


## Pixel-wise RMSE maps — GT vs the 4 methods (mean over 25 sequences)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PIXEL-WISE RMSE MAPS — Ground Truth vs the 4 methods, mean over the 25 sequences
# Per-pixel RMSE across the 25 sequences, one figure per forecast lead time.
# Native polar-stereographic grid (xc/yc) → artefact-free maps.
# ══════════════════════════════════════════════════════════════════════════════
import time as _t
from tqdm.auto import tqdm
import numpy as np, xarray as xr, matplotlib.pyplot as plt

print("=" * 80); print("PIXEL-WISE RMSE — accumulating over 25 sequences"); print("=" * 80)
t0 = _t.time()

rmse_maps = {'x50': {}, 'x10': {}}     # rmse_maps[res][method] = (N_FORECAST, ny, nx)
rmse_grid = {}                          # rmse_grid[res] = (xc, yc)

for res in ['x10', 'x50', 'x2']:
    methods_res = [m for m in METHODS if res in METHODS[m]['res']]
    for method in methods_res:
        sse = cnt = None
        for idx, *_ in tqdm(SEQUENCES, desc=f'RMSE {res}/{method}', leave=False):
            fp = method_file(method, idx, res)
            if fp is None: continue
            with xr.open_dataset(fp) as ds:
                pred = ds['pred_SIC'].isel(time=slice(-N_FORECAST, None)).values
                gt   = ds[GT_VAR[res]].isel(time=slice(-N_FORECAST, None)).values
                if res not in rmse_grid:
                    rmse_grid[res] = (ds.xc.values, ds.yc.values)
            err2 = (pred - gt) ** 2
            valid = np.isfinite(err2)
            if sse is None:
                sse = np.zeros_like(err2); cnt = np.zeros_like(err2)
            sse = np.where(valid, sse + np.where(valid, err2, 0.0), sse)
            cnt += valid
        rmse_maps[res][method] = np.sqrt(np.divide(sse, cnt, out=np.full_like(sse, np.nan), where=cnt > 0))
        print(f"  {res} / {method}: done")
print(f"accumulation in {_t.time()-t0:.1f}s")

# ── Plot — one figure per lead time, methods in a 2×2 grid, shared colorbar ────
for res in ['x10', 'x50', 'x2']:
    methods_res = [m for m in METHODS if res in METHODS[m]['res']]
    xc, yc = rmse_grid[res]
    vmax = float(np.nanpercentile(np.concatenate([rmse_maps[res][m].ravel() for m in methods_res]), 99))
    nrow = 2 if len(methods_res) > 2 else 1
    # nrow-adaptive margins so the suptitle never overlaps the subplot titles
    # (the 1-row x50 case needs more headroom than the 2-row x10 case).
    fig_h   = 5.0 * nrow if nrow > 1 else 5.8
    top, bottom, cbar_y = (0.91, 0.09, 0.045) if nrow > 1 else (0.82, 0.17, 0.09)

    for lt in range(N_FORECAST):
        fig, axes = plt.subplots(nrow, 2, figsize=(8.6, fig_h), subplot_kw={'projection': DATA_CRS})
        axes = np.atleast_1d(axes).ravel()
        pcm = None
        for ax, method in zip(axes, methods_res):
            pcm = polar_map(ax, xc, yc, rmse_maps[res][method][lt], cmap='inferno', vmin=0, vmax=vmax)
            ax.set_title(method, fontsize=12, fontweight='bold')
        for ax in axes[len(methods_res):]:
            ax.axis('off')
        cax = fig.add_axes([0.25, cbar_y, 0.5, 0.018])
        fig.colorbar(pcm, cax=cax, orientation='horizontal').set_label(
            'Pixel-wise RMSE vs Ground Truth', fontsize=11)
        fig.suptitle(f'Sea Ice Concentration forecast RMSE — {res}  |  Lead-{lt}  '
                     f'(mean over {len(SEQUENCES)} sequences)', fontsize=13, y=0.985)
        fig.subplots_adjust(left=0.02, right=0.98, top=top, bottom=bottom, wspace=0.05, hspace=0.14)
        plt.show()
print(f"\nRMSE maps done in {_t.time()-t0:.1f}s")


## 1. Segments Class for Cartesian Coordinates

`SegmentsCartesian` generates random transects on a regular x/y (polar stereographic) grid using Euclidean geometry and Bresenham's line algorithm.


In [ ]:
class SegmentsCartesian:
    """
    Segments class adapted for Cartesian x/y coordinate systems.
    Designed for Arctic grids with regular spacing in x/y (e.g., polar stereographic).
    
    Parameters
    ----------
    num_segments : int
        Number of random segments to generate
    min_dist : float
        Minimum segment length in kilometers
    max_dist : float
        Maximum segment length in kilometers
    overlapping : float
        Overlap fraction between segments (0-1)
    dx : float
        Grid spacing in x-direction (kilometers)
    dy : float
        Grid spacing in y-direction (kilometers)
    """
    
    def __init__(self, num_segments=1000, min_dist=200, max_dist=1000, 
                 overlapping=0.25, dx=2.2, dy=2.2):
        self.num_segments = num_segments
        self.min_dist = min_dist
        self.max_dist = max_dist
        self.overlapping = overlapping
        self.dx = dx  # Grid spacing in km
        self.dy = dy  # Grid spacing in km
        
    def find_point_cartesian(self, point_idx, angle, dist_km):
        """
        Find endpoint in Cartesian coordinates.
        
        Parameters
        ----------
        point_idx : tuple
            Starting point as (x_idx, y_idx) grid indices
        angle : float
            Angle in degrees (0° = East, 90° = North)
        dist_km : float
            Distance in kilometers
            
        Returns
        -------
        tuple
            Endpoint as (x_idx, y_idx) grid indices
        """
        x_idx, y_idx = point_idx
        angle_rad = np.radians(angle)
        
        # Convert distance to grid indices
        dx_cells = (dist_km * np.cos(angle_rad)) / self.dx
        dy_cells = (dist_km * np.sin(angle_rad)) / self.dy
        
        x_end = int(x_idx + dx_cells)
        y_end = int(y_idx + dy_cells)
        
        return x_end, y_end
    
    def get_idx_pt_endline(self, x1, y1, theta, distance_km):
        """
        Get endpoint indices for a line segment.
        
        Parameters
        ----------
        x1, y1 : int
            Starting grid indices
        theta : float
            Angle in degrees
        distance_km : float
            Distance in kilometers
            
        Returns
        -------
        tuple
            (x2, y2) endpoint grid indices
        """
        return self.find_point_cartesian((x1, y1), theta, distance_km)
    
    def compute_spatial_segment(self, time, x, y, obs, rec, start_mask=None):
        """
        Compute spatial segments on Cartesian grid.

        start_mask : 2-D bool array (Ny, Nx), optional
            If given, segment start points are restricted to True pixels
            (e.g. Marginal Ice Zone).  Dramatically speeds up generation
            when most of the domain is ocean / NaN.
        
        Parameters
        ----------
        time : array
            Time coordinates
        x : array
            X-coordinates (1D array, can be in any units)
        y : array
            Y-coordinates (1D array, can be in any units)
        obs : array
            Observation data (time, y, x)
        rec : array
            Reconstruction data (time, y, x)
            
        Returns
        -------
        tuple
            (time_segments, x_segments, y_segments, obs_segments, rec_segments)
        """
        Nx = len(x)
        Ny = len(y)
        Nt = len(time)
        mesh_x, mesh_y = np.meshgrid(x, y)
        mesh_x = mesh_x.T   # shape (Nx, Ny)
        mesh_y = mesh_y.T

        # MIZ: pre-compute valid (iy, ix) pairs if a mask is provided
        if start_mask is not None:
            _vy, _vx = np.where(start_mask)      # shape (Ny,Nx)
            if len(_vx) == 0:
                raise ValueError("start_mask has no valid pixels — check MIZ thresholds")

        time_segments = []
        x_segments = []
        y_segments = []
        obs_segments = []
        rec_segments = []
        
        n_segments = 0
        while n_segments < self.num_segments:
            # Generate random starting points (MIZ-constrained if mask given)
            if start_mask is not None:
                _sel = np.random.randint(len(_vx), size=1000)
                start_x = _vx[_sel]   # xc indices  (Nx dimension)
                start_y = _vy[_sel]   # yc indices  (Ny dimension)
            else:
                start_x = np.random.randint(Nx, size=1000)
                start_y = np.random.randint(Ny, size=1000)
            theta = np.random.randint(360, size=1000)
            distance = np.random.randint(self.min_dist, self.max_dist + 100, size=1000)
            
            # Calculate endpoints
            x2_y2 = [self.get_idx_pt_endline(x1, y1, angle, dist) 
                     for x1, y1, angle, dist in zip(start_x, start_y, theta, distance)]
            end_x = [xy[0] for xy in x2_y2]
            end_y = [xy[1] for xy in x2_y2]
            
            # Generate overlapping segments
            ovlp_distance = distance * (1 - self.overlapping)
            ovlp_start_x_start_y = [self.get_idx_pt_endline(x1, y1, angle, dist) 
                                    for x1, y1, angle, dist in 
                                    zip(start_x, start_y, theta, ovlp_distance)]
            ovlp_start_x = [xy[0] for xy in ovlp_start_x_start_y if xy[0] < Nx]
            ovlp_start_y = [xy[1] for xy in ovlp_start_x_start_y if xy[1] < Ny]
            
            ovlp_end_x_end_y = [self.get_idx_pt_endline(x1, y1, angle, dist) 
                                for x1, y1, angle, dist in 
                                zip(ovlp_start_x, ovlp_start_y, theta, distance)]
            ovlp_end_x = [xy[0] for xy in ovlp_end_x_end_y]
            ovlp_end_y = [xy[1] for xy in ovlp_end_x_end_y]
            
            # Concatenate candidates
            start_x = np.concatenate((start_x, ovlp_start_x))
            start_y = np.concatenate((start_y, ovlp_start_y))
            end_x = np.concatenate((end_x, ovlp_end_x))
            end_y = np.concatenate((end_y, ovlp_end_y))
            
            # Generate line coordinates
            line_in_img = zip(start_x, start_y, end_x, end_y)
            rr_cc = [line(*l) for l in line_in_img]
            rr = [c[0] for c in rr_cc]
            cc = [c[1] for c in rr_cc]
            # Random time selection
            tt = np.random.randint(Nt, size=len(cc))
            
            # Filter valid segments - check ALL points are in bounds
            def is_valid_segment(r, c):
                """Check if all points in segment are within bounds"""
                return (np.all(r >= 0) and np.all(r < Nx) and 
                       np.all(c >= 0) and np.all(c < Ny))
            
            valid_mask = [is_valid_segment(r, c) for r, c in zip(rr, cc)]
            
            time_segment = [t for t, valid in zip(tt, valid_mask) if valid]
            x_segment = [mesh_x[r, c] for r, c, valid in zip(rr, cc, valid_mask) if valid]
            y_segment = [mesh_y[r, c] for r, c, valid in zip(rr, cc, valid_mask) if valid]
            obs_segment = [obs[t, c, r] for t, r, c, valid in zip(tt, rr, cc, valid_mask) if valid]
            rec_segment = [rec[t, c, r] for t, r, c, valid in zip(tt, rr, cc, valid_mask) if valid]
            
            if len(obs_segment) > 0:
                # Check for NaN values
                ratio_nan_obs = [100 * (1 - np.mean(np.isfinite(s))) for s in obs_segment]
                ratio_nan_rec = [100 * (1 - np.mean(np.isfinite(s))) for s in rec_segment]
                
                # Select valid segments
                idx_valid = [i for i in range(len(obs_segment)) 
                            if (ratio_nan_obs[i] == 0. and 
                                ratio_nan_rec[i] == 0. and 
                                len(x_segment[i]) >= self.min_dist / self.dx)]
                
                if len(idx_valid) > 0:
                    time_segments += [time_segment[i] for i in idx_valid]
                    x_segments += [x_segment[i] for i in idx_valid]
                    y_segments += [y_segment[i] for i in idx_valid]
                    obs_segments += [obs_segment[i] for i in idx_valid]
                    rec_segments += [rec_segment[i] for i in idx_valid]
            
            n_segments = len(x_segments)
            if n_segments % 100 == 0:
                print(f"Generated {n_segments} segments")
        
        # Standardize segment lengths
        nperseg = np.min([len(x_segments[i]) for i in range(len(x_segments))])
        x_segments = [x_segments[i][:nperseg] for i in range(len(x_segments))]
        y_segments = [y_segments[i][:nperseg] for i in range(len(y_segments))]
        obs_segments = [obs_segments[i][:nperseg] for i in range(len(obs_segments))]
        rec_segments = [rec_segments[i][:nperseg] for i in range(len(rec_segments))]
        
        return time_segments, x_segments, y_segments, obs_segments, rec_segments
        
    def compute_time_segment(self, time, x, y, obs, rec, criteria="rec"):
        """
        Compute temporal segments on Cartesian grid.
        
        Parameters
        ----------
        time : array
            Time coordinates
        x : array
            X-coordinates
        y : array
            Y-coordinates
        obs : array
            Observation data (time, y, x)
        rec : array
            Reconstruction data (time, y, x)
        criteria : str
            Validation criteria ('rec' or 'obs')
            
        Returns
        -------
        tuple
            (time_segments, x_segments, y_segments, obs_segments, rec_segments)
        """
        Nx = len(x)
        Ny = len(y)
        mesh_x, mesh_y = np.meshgrid(x, y)
        mesh_x = mesh_x.T
        mesh_y = mesh_y.T
        Nt = len(time)
        mesh_time = np.arange(Nt)
        
        time_segments = []
        x_segments = []
        y_segments = []
        obs_segments = []
        rec_segments = []
        
        n_segments = 0
        while n_segments < self.num_segments:
            start_t = np.random.randint(Nt, size=1000)
            distance = np.random.randint(self.max_dist, self.max_dist + 5, size=1000)
            end_t = [t + d for t, d in zip(start_t, np.concatenate((distance, distance)))]
            # Overlapping segments
            ovlp_t = [t + int(d * (1 - self.overlapping)) for t, d in zip(start_t, distance)]
            
            # Concatenate
            start_t = np.concatenate((start_t, ovlp_t))
            end_t = [t + d for t, d in zip(start_t, np.concatenate((distance, distance)))]
            time_segment = [mesh_time[ts:te] for ts, te in zip(start_t, end_t) if te < Nt]
            # Random spatial positions
            x_idx = np.random.randint(Nx, size=len(start_t))
            y_idx = np.random.randint(Ny, size=len(start_t))
            
            # Generate segments
            time_segment = [mesh_time[ts:te] for ts, te in zip(start_t, end_t) if te < Nt]
            x_segment = [mesh_x[r, c] for ts, te, r, c in zip(start_t, end_t, x_idx, y_idx) if te < Nt]
            y_segment = [mesh_y[r, c] for ts, te, r, c in zip(start_t, end_t, x_idx, y_idx) if te < Nt]
            obs_segment = [obs[ts:te, c, r] for ts, te, r, c in zip(start_t, end_t, x_idx, y_idx) if te < Nt]
            rec_segment = [rec[ts:te, c, r] for ts, te, r, c in zip(start_t, end_t, x_idx, y_idx) if te < Nt]
            
            if len(obs_segment) > 0:
                ratio_nan_obs = [100 * (1 - np.mean(np.isfinite(s))) for s in obs_segment]
                ratio_nan_rec = [100 * (1 - np.mean(np.isfinite(s))) for s in rec_segment]
                
                if criteria != "rec":
                    idx_valid = [i for i in range(len(obs_segment)) 
                                if (ratio_nan_obs[i] == 0. and 
                                    ratio_nan_rec[i] == 0. and 
                                    len(time_segment[i]) >= self.max_dist)]
                else:
                    idx_valid = [i for i in range(len(obs_segment)) 
                                if (ratio_nan_rec[i] == 0. and 
                                    len(time_segment[i]) >= self.max_dist)]
                
                if len(idx_valid) > 0:
                    time_segments += [time_segment[i] for i in idx_valid]
                    x_segments += [x_segment[i] for i in idx_valid]
                    y_segments += [y_segment[i] for i in idx_valid]
                    obs_segments += [obs_segment[i] for i in idx_valid]
                    rec_segments += [rec_segment[i] for i in idx_valid]
            

            n_segments = len(x_segments)

            if n_segments % 100 == 0:
                print(f"Generated {n_segments} segments")
        
        # Standardize
        nperseg = np.min([len(time_segments[i]) for i in range(len(time_segments))])
        time_segments = [time_segments[i][:nperseg] for i in range(len(time_segments))]
        obs_segments = [obs_segments[i][:nperseg] for i in range(len(obs_segments))]
        rec_segments = [rec_segments[i][:nperseg] for i in range(len(rec_segments))]
        return time_segments, x_segments, y_segments, obs_segments, rec_segments

## 2. Spatial Segments — sampling & visualisation

Compute PSD scores over the **analysis period** (non-forecast days) for both resolutions.  
The reference is `models_SIC` and the reconstruction is `pred_SIC`.


In [ ]:
# ── Prepare data for spectral analysis — analysis period ──────────────────────
print("=" * 80)
print("PREPARING DATA FOR SPECTRAL ANALYSIS (ANALYSIS PERIOD)")
print("=" * 80)

spectral_data = {}
time_coord_dummy = np.arange(1)   # single-time wrapper for segment generator

for key in ['x50', 'x10', 'x2']:
    d = cristal_data[key]
    nt_a = d['pred_analysis'].shape[0]

    spectral_data[key] = {
        'time': np.arange(nt_a),
        'x': d['xc'],
        'y': d['yc'],
        'obs': d['models_analysis'],   # (time, yc, xc)
        'rec': d['pred_analysis'],
        'dx': d['dx'],
        'dy': d['dy'],
        'resolution_km': d['resolution_km'],
    }

    print(f"\n✓ {key} — {d['resolution_km']:.0f} km")
    print(f"  Analysis shape : {d['pred_analysis'].shape} (time, yc, xc)")
    print(f"  Grid spacing   : dx={d['dx']:.1f} km, dy={d['dy']:.1f} km")

print("\n" + "=" * 80)
print("✓ DATASETS READY")
print("=" * 80)


In [ ]:
# ── Generate spatial segments — analysis period ────────────────────────────────
print("\n" + "=" * 80)
print("GENERATING SPATIAL SEGMENTS (ANALYSIS PERIOD)")
print("=" * 80)

# Minimum latitude for segment start points (degrees N)
# Restricts segments to the sea-ice zone above Greenland
LAT_MIN_SEG = 70.0

# Resolution-aware segment length bounds (target ~20–80× Nyquist wavelength)
SEGMENT_PARAMS = {
    'x50': {'min_dist': 500, 'max_dist': 2000, 'num_segments': 300},
    'x10': {'min_dist': 100, 'max_dist':  800, 'num_segments': 300},
    'x2':  {'min_dist':  20, 'max_dist':  160, 'num_segments': 300},
}

segments_data = {}

for key, data in spectral_data.items():
    p = SEGMENT_PARAMS[key]
    print(f"\n{key} — {data['resolution_km']:.0f} km  "
          f"(segments {p['min_dist']}–{p['max_dist']} km, n={p['num_segments']})")

    seg = SegmentsCartesian(
        num_segments=p['num_segments'],
        min_dist=p['min_dist'],
        max_dist=p['max_dist'],
        dx=data['dx'],
        dy=data['dy'],
    )
    try:
        d = cristal_data[key]
        # Mask: only start segments in the Marginal Ice Zone (MIZ) above LAT_MIN_SEG
        # (0.15 < mean SIC < 0.85 over analysis period — the ice-edge transition)
        sic_mean   = np.nanmean(d['models_analysis'], axis=0)   # (Ny, Nx)
        cov        = np.mean(np.isfinite(d['models_analysis']), axis=0)   # obs coverage (1.0 for model GT)
        start_mask = (sic_mean > 0.15) & (sic_mean < 0.85) & (cov > 0.5) & (d['lat'] > LAT_MIN_SEG)
        n_valid = int(start_mask.sum())
        print(f'  Mask: {n_valid:,} valid start pixels  '
              f'({100*n_valid/start_mask.size:.1f}% of grid)  '
              f'— lat>{LAT_MIN_SEG}° & MIZ (0.15<SIC<0.85) & obs-cov>50%')
        t_s, x_s, y_s, obs_s, rec_s = seg.compute_spatial_segment(
            data['time'], data['x'], data['y'], data['obs'], data['rec'],
            start_mask=start_mask
        )
        segments_data[key] = {
            'time': t_s, 'x': x_s, 'y': y_s, 'obs': obs_s, 'rec': rec_s,
            'count': len(x_s),
            'length': len(x_s[0]) if x_s else 0,
        }
        print(f"  ✓ {len(x_s)} segments  |  length = {len(x_s[0])} pts "
              f"(≈ {len(x_s[0]) * data['dx']:.0f} km)")
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        segments_data[key] = None

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)


In [ ]:
# ── Visualise sampled segments on the last analysis day — x50, native grid ────
import numpy as np, matplotlib.pyplot as plt
# Plotted in the native polar-stereographic grid (xc/yc) -> no projection distortion.
fig, ax = plt.subplots(figsize=(8, 8.4), subplot_kw={'projection': DATA_CRS})

d = cristal_data['x50']
xc, yc = d['xc'], d['yc']
im = polar_map(ax, xc, yc, d['models_analysis'][-1], cmap='viridis', vmin=0, vmax=1)

# Overlay segments directly in native xc/yc (metres): white lines, diamond end
# markers (white interior, black edge) at both ends.
seg = segments_data.get('x50')
n_count = 0
if seg is not None and seg['count'] > 0:
    for i in range(min(40, seg['count'])):
        xs = np.asarray(seg['x'][i]); ys = np.asarray(seg['y'][i])
        ax.plot(xs, ys, '-', color='white', lw=1.8, alpha=0.85, transform=DATA_CRS, zorder=5)
        for xe, ye in [(xs[0], ys[0]), (xs[-1], ys[-1])]:
            ax.plot(xe, ye, marker='D', ms=6, mfc='white', mec='black', mew=1.0,
                    transform=DATA_CRS, zorder=6)
    n_count = seg['count']

ax.set_title(f"x50 — {d['resolution_km']:.0f} km  ({n_count} segments shown)",
             fontsize=12, fontweight='bold')
cax = fig.add_axes([0.2, 0.06, 0.6, 0.025])
fig.colorbar(im, cax=cax, orientation='horizontal').set_label('Sea Ice Concentration', fontsize=11)
fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.13)
fig.suptitle('Spatial segments — analysis period (MIZ, lat > 70 N)',
             fontsize=13, fontweight='bold', y=0.98)
out = '/data/users/maxb/croscim_sic_segments.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f"\nSaved to {out}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SPECTRAL + μ AGGREGATION over the 25 sequences (forecast only, per sub-region)
#   • Fixed spatial segments per (resolution, region) — generated once, reused for
#     every sequence so the PSD wavenumber grid is identical and poolable.
#   • PSD pooled over all 25 sequences  ⇒  average PSD-based metrics by lead time.
#   • psd_fc_regions[res][region][lt] holds the PLOT_METHOD curves (used by the
#     3×2 spectral plot & resolved-scale cells).
#   • bench_* hold per-method metrics (μ-score, resolved-km) for the score table.
# ══════════════════════════════════════════════════════════════════════════════
import scipy.signal, time as _time
from tqdm.auto import tqdm
import numpy as np, xarray as xr

FC_REGIONS = {
    'Global':           None,
    'Barents Sea':      [15,  70, 70, 82],
    'Kara Sea':         [60, 100, 70, 82],
    'Greenland / Fram': [-20, 20, 70, 85],
}
FC_REGION_COLORS = {'Global': '#333333', **REGION_COLORS}
FC_SEGMENT_PARAMS = {
    'x50': dict(num_segments=1500, min_dist=500, max_dist=2000),
    'x10': dict(num_segments=1500, min_dist=100, max_dist=800),
    'x2':  dict(num_segments=1500, min_dist=20,  max_dist=160),
}

def _extract_fc(field_2d, xc, yc, x_paths, y_paths):
    dxm = float(xc[1]-xc[0]); dym = float(yc[1]-yc[0])
    segs = []
    for xp, yp in zip(x_paths, y_paths):
        ix = np.clip(np.round((xp-xc[0])/dxm).astype(int), 0, len(xc)-1)
        iy = np.clip(np.round((yp-yc[0])/dym).astype(int), 0, len(yc)-1)
        s = field_2d[iy, ix]
        if np.all(np.isfinite(s)):
            segs.append(s)
    return segs

def _welch_pool(obs_segs, rec_segs, delta_km):
    if len(obs_segs) < 5: return None
    arr_o = np.asarray(obs_segs); arr_r = np.asarray(rec_segs)
    kw = dict(fs=1.0/delta_km, nperseg=arr_o.shape[1],
              scaling='density', detrend='linear', noverlap=0)
    wn, p_ref   = scipy.signal.welch(arr_o.flatten(), **kw)
    _,  p_study = scipy.signal.welch(arr_r.flatten(), **kw)
    _,  p_diff  = scipy.signal.welch((arr_r-arr_o).flatten(), **kw)
    return wn, p_ref, p_study, p_diff

def _cross05(wn, p_ref, p_diff, thresh=0.5):
    wl = 1.0/np.where(wn > 0, wn, np.nan)
    score = 1.0 - p_diff/p_ref
    m = np.isfinite(wl) & np.isfinite(score)
    wl, score = wl[m], score[m]
    for i in range(len(score)-1):
        if (score[i]-thresh)*(score[i+1]-thresh) < 0:
            t = (thresh-score[i])/(score[i+1]-score[i])
            return float(np.exp(np.log(wl[i]) + t*(np.log(wl[i+1])-np.log(wl[i]))))
    return np.nan

def _region_mu(pred_2d, gt_2d, lat, lon, ext):
    m = np.isfinite(pred_2d) & np.isfinite(gt_2d)
    if ext is not None:
        lo0, lo1, la0, la1 = ext
        m &= (lat >= la0) & (lat <= la1) & (lon >= lo0) & (lon <= lo1)
    if m.sum() < 10: return np.nan
    g = gt_2d[m]; p = pred_2d[m]
    sd = np.std(g)
    if sd < 1e-9: return np.nan
    return 1.0 - float(np.sqrt(np.mean((p-g)**2)) / sd)

print("=" * 80)
print("SPECTRAL + μ AGGREGATION over 25 sequences")
print("=" * 80)
t0 = _time.time()

# ── 1. Fixed segment paths per (res, region) from seq01 GT (PLOT_METHOD) ──────
seg_paths = {'x50': {}, 'x10': {}}
for res in ['x50', 'x10', 'x2']:
    d = cristal_data[res]            # seq01, PLOT_METHOD (from the load cell)
    sic_mean = np.nanmean(d['models_analysis'], axis=0)
    cov      = np.mean(np.isfinite(d['models_analysis']), axis=0)   # obs coverage (1.0 for model GT)
    p = FC_SEGMENT_PARAMS[res]
    for rname, ext in FC_REGIONS.items():
        miz = (sic_mean > 0.15) & (sic_mean < 0.85) & (cov > 0.5)   # MIZ in well-observed pixels
        if ext is None:
            start_mask = miz & (d['lat'] > 65.0)
        else:
            lo0, lo1, la0, la1 = ext
            start_mask = miz & (d['lat'] >= la0) & (d['lat'] <= la1) & \
                         (d['lon'] >= lo0) & (d['lon'] <= lo1)
        if int(start_mask.sum()) < 10:
            seg_paths[res][rname] = ([], []); continue
        seg = SegmentsCartesian(num_segments=p['num_segments'],
                                min_dist=p['min_dist'], max_dist=p['max_dist'],
                                dx=d['dx'], dy=d['dx'])
        _, xs, ys, _, _ = seg.compute_spatial_segment(
            np.arange(d['models_analysis'].shape[0]), d['xc'], d['yc'],
            d['models_analysis'], d['models_analysis'], start_mask=start_mask)
        seg_paths[res][rname] = (xs, ys)
    print(f"  {res}: segment paths " +
          ", ".join(f"{r}={len(seg_paths[res][r][0])}" for r in FC_REGIONS))

# ── 2. Pool segments + μ over the 25 sequences, per method / res / region / lt ─
pooled = {}     # pooled[(method,res,region,lt)] = ([obs...],[rec...])
mu_acc = {}     # mu_acc[(method,res,region,lt)] = [per-seq μ ...]

for res in ['x10', 'x50', 'x2']:
    methods_res = [m for m in METHODS if res in METHODS[m]['res']]
    grid_lat = grid_lon = grid_xc = grid_yc = None
    for method in methods_res:
        for idx, *_ in tqdm(SEQUENCES, desc=f'PSD {res}/{method}', leave=False):
            fp = method_file(method, idx, res)
            if fp is None: continue
            with xr.open_dataset(fp) as ds:
                pred = ds['pred_SIC'].isel(time=slice(-N_FORECAST, None)).values
                gt   = ds[GT_VAR[res]].isel(time=slice(-N_FORECAST, None)).values
                if grid_lat is None:
                    grid_lat = ds['lat'].values
                    grid_lon = (ds['lon'].values + 180) % 360 - 180
                    grid_xc, grid_yc = ds.xc.values, ds.yc.values
            for rname, ext in FC_REGIONS.items():
                xp, yp = seg_paths[res][rname]
                for lt in range(N_FORECAST):
                    key = (method, res, rname, lt)
                    if xp:
                        o = _extract_fc(gt[lt],   grid_xc, grid_yc, xp, yp)
                        r = _extract_fc(pred[lt], grid_xc, grid_yc, xp, yp)
                        po, pr = pooled.setdefault(key, ([], []))
                        po.extend(o); pr.extend(r)
                    mu = _region_mu(pred[lt], gt[lt], grid_lat, grid_lon, ext)
                    if np.isfinite(mu):
                        mu_acc.setdefault(key, []).append(mu)
        print(f"  pooled {res} / {method}")

# ── 3. Reduce: PSD from pooled segments, resolved-km, mean μ ───────────────────
psd_fc_regions = {'x50': {r: {} for r in FC_REGIONS},
                  'x10': {r: {} for r in FC_REGIONS},
                  'x2':  {r: {} for r in FC_REGIONS}}     # PLOT_METHOD curves
bench_psd      = {}     # bench_psd[method][res][region][lt] = (wn,p_ref,p_study,p_diff)
bench_resolved = {}     # bench_resolved[method][res][region][lt] = km
bench_mu       = {}     # bench_mu[method][res][region][lt]       = mean μ

for (method, res, rname, lt), (po, pr) in pooled.items():
    delta = cristal_data[res]['dx']
    psd = _welch_pool(po, pr, delta)
    bench_psd.setdefault(method, {}).setdefault(res, {}).setdefault(rname, {})[lt] = psd
    bench_resolved.setdefault(method, {}).setdefault(res, {}).setdefault(rname, {})[lt] = (
        _cross05(psd[0], psd[1], psd[3]) if psd else np.nan)

for key, vals in mu_acc.items():
    method, res, rname, lt = key
    bench_mu.setdefault(method, {}).setdefault(res, {}).setdefault(rname, {})[lt] = float(np.mean(vals))

# Pick the BEST method (highest mean μ over Global region & lead times, x10) for the
# spectral display, and populate psd_fc_regions from its pooled PSD.
# Experiment (xp) shown in the spectral plot — change here to switch method.
SPECTRAL_XP = 'UNet-UOAI'
def _meanmu(m):
    g = bench_mu.get(m, {}).get('x10', {}).get('Global', {})
    return np.mean(list(g.values())) if g else -np.inf
BEST_METHOD = SPECTRAL_XP if SPECTRAL_XP in bench_psd else max(METHODS, key=_meanmu)
for res in ['x50', 'x10', 'x2']:
    for rname in FC_REGIONS:
        for lt, psd in bench_psd.get(BEST_METHOD, {}).get(res, {}).get(rname, {}).items():
            psd_fc_regions[res][rname][lt] = psd

print(f"\n✓ aggregation done in {_time.time()-t0:.1f}s  "
      f"(best method for spectral plot = {BEST_METHOD})")


## 3. Forecast Spectral Score Plots — Per Lead Time & Resolution


In [ ]:
# ── Forecast PSD plots — sub-domains × 3 resolutions (mean over all sequences) ─
# Layout : 3 rows (lead-0/1/2) × 2 cols (PSD power | PSD score)
# Colour : sub-region (harmonised palette).  Width : resolution (thick x2 / thin x50)
# Style  : solid = Reference, dashed = ML model.  Markers : per-region (black edge).
# xp     : experiment plotted (reads bench_psd[xp]). Change `xp` below to switch.
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

_REGIONS = ['Global', 'Barents Sea', 'Kara Sea', 'Greenland / Fram']
_REGION_LABEL = {'Global': 'Whole Arctic', 'Barents Sea': 'Barents Sea',
                 'Kara Sea': 'Kara Sea', 'Greenland / Fram': 'Greenland / Fram'}
_COLORS  = {'Global': '#333333', **REGION_COLORS}      # harmonised, single source
_MARKERS = {'Global': 'o', 'Barents Sea': 's', 'Kara Sea': '^', 'Greenland / Fram': 'D'}
_RES_LW  = {'x50': 1.0, 'x10': 2.0, 'x2': 3.2}         # thin = x50 … thick = x2
_NYQ_COLOR = {'x50': '#2166AC', 'x10': '#D6604D', 'x2': '#1A9850'}
SCORE_THRESH = 0.5
_FS = 11
xp = 'UNet-UOAI'           # <-- experiment plotted in THIS cell


def _cross_05(wl, score):
    """First wavelength where the score crosses 0.5 (one value per curve)."""
    m = np.isfinite(wl) & np.isfinite(score)
    wl, score = wl[m], score[m]
    for i in range(len(score) - 1):
        if (score[i] - SCORE_THRESH) * (score[i + 1] - SCORE_THRESH) < 0:
            t = (SCORE_THRESH - score[i]) / (score[i + 1] - score[i])
            return float(np.exp(np.log(wl[i]) + t * (np.log(wl[i + 1]) - np.log(wl[i]))))
    return None


def _mk(ax, x, y, *, col, mrk, lw, ls, log=False):
    """Line with region-coloured markers (black edge), markers thinned out."""
    plot = ax.loglog if log else ax.semilogx
    me = max(1, len(x) // 7)
    plot(x, y, color=col, lw=lw, ls=ls, marker=mrk, markevery=me,
         ms=6, mfc=col, mec='black', mew=0.8, zorder=3, label='_nolegend_')


# Share x (wavelength) across all; share y per column (PSD col / Score col)
fig, axes = plt.subplots(3, 2, figsize=(15, 6.2 * 3),
                         sharex='all', sharey='col', constrained_layout=False)
fig.subplots_adjust(hspace=0.55, wspace=0.22, bottom=0.13, top=0.88)

for lt in range(3):
    ax_psd, ax_score = axes[lt, 0], axes[lt, 1]
    cut_entries = []                      # one (wavelength, colour) per curve

    for rname in _REGIONS:
        col, mrk = _COLORS[rname], _MARKERS[rname]
        for key in ['x50', 'x10', 'x2']:
            psd = bench_psd.get(xp, {}).get(key, {}).get(rname, {}).get(lt)
            if psd is None:
                continue
            wn, p_ref, p_study, p_diff = psd
            wl = 1.0 / np.where(wn > 0, wn, np.nan)
            score = 1.0 - p_diff / p_ref
            lw = _RES_LW[key]
            _mk(ax_psd, wl, p_ref,   col=col, mrk=mrk, lw=lw, ls='-',  log=True)
            _mk(ax_psd, wl, p_study, col=col, mrk=mrk, lw=lw, ls='--', log=True)
            _mk(ax_score, wl, score, col=col, mrk=mrk, lw=lw, ls='-')
            rs_km = _cross_05(wl, score)          # first crossing only -> one per curve
            if rs_km is not None:
                ax_score.axvline(rs_km, color=col, lw=0.9, ls=':', alpha=0.55,
                                 zorder=2, label='_nolegend_')
                cut_entries.append((rs_km, col))

    for key in ['x50', 'x10', 'x2']:
        nyq = 2.0 * cristal_data[key]['resolution_km']
        for ax in (ax_psd, ax_score):
            ax.axvline(nyq, color=_NYQ_COLOR[key], lw=1.1, ls='-.', alpha=0.55,
                       zorder=2, label='_nolegend_')
    ax_score.axhline(SCORE_THRESH, color='black', lw=1.6, ls='--', alpha=0.75,
                     zorder=2, label='_nolegend_')

    for ax in (ax_psd, ax_score):
        ax.grid(True, which='both', alpha=0.18, ls='--')
        ax.tick_params(labelsize=_FS - 1)
    ax_psd.set_ylabel('PSD  (km)', fontsize=_FS)
    ax_score.set_ylabel('PSD Score  [1 − err/ref]', fontsize=_FS)

    # 0.5-cut wavelengths: on the TOP edge of the score box, 45°, small font,
    # 2-level stagger to avoid overlap; titles are padded up to clear them.
    trans = ax_score.get_xaxis_transform()
    for j, (rs_km, col) in enumerate(sorted(cut_entries)):
        ax_score.annotate(f'{rs_km:.0f}', xy=(rs_km, 1.0), xytext=(rs_km, 1.02 + 0.07 * (j % 2)),
                          xycoords=trans, textcoords=trans, rotation=45,
                          ha='left', va='bottom', fontsize=8.5, color=col,
                          fontweight='bold', annotation_clip=False)

    ax_psd.set_title(f'Lead-{lt} — PSD', fontsize=_FS, fontweight='bold', pad=30)
    ax_score.set_title(f'Lead-{lt} — Score', fontsize=_FS, fontweight='bold', pad=30)

# x shared -> invert once; ylim of the Score column set once (shared)
axes[0, 0].invert_xaxis()
axes[0, 1].set_ylim([-0.18, 1.05])
for ax in (axes[2, 0], axes[2, 1]):
    ax.set_xlabel('Wavelength (km)', fontsize=_FS)

# ── Legend on TWO lines: regions (row 1) then styles (row 2) ──────────────────
_region_legend = [
    mlines.Line2D([], [], color=_COLORS[r], ls='-', lw=2.4, marker=_MARKERS[r],
                  ms=7, mfc=_COLORS[r], mec='black', mew=0.8, label=_REGION_LABEL[r])
    for r in _REGIONS
]
_style_legend = [
    mlines.Line2D([], [], color='#555555', ls='-',  lw=2.4, label='Reference'),
    mlines.Line2D([], [], color='#555555', ls='--', lw=2.4, label='ML model'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=3.2, label='x2 (1 km)'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=2.0, label='x10 (5 km)'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=1.0, label='x50 (25 km)'),
    mlines.Line2D([], [], color='black',   ls='--', lw=1.5, alpha=0.8, label='Score = 0.5'),
]
fig.legend(handles=_region_legend, loc='lower center', bbox_to_anchor=(0.5, 0.075),
           ncols=len(_region_legend), frameon=True, fontsize=_FS - 0.5,
           handlelength=2.2, columnspacing=1.4)
fig.legend(handles=_style_legend, loc='lower center', bbox_to_anchor=(0.5, 0.028),
           ncols=len(_style_legend), frameon=True, fontsize=_FS - 0.5,
           handlelength=2.2, columnspacing=1.4)

fig.suptitle(
    f'{xp} — Forecast PSD score (mean over {len(SEQUENCES)} sequences)\n'
    '(solid = reference,  dashed = ML model,  thick = x2 / thin = x50)',
    fontsize=_FS + 1, fontweight='bold', y=0.985)
plt.show()


### Same spectral plot for the res2 experiment (UNet-UOAI-res2)

In [ ]:
# ── Forecast PSD plots — sub-domains × 3 resolutions (mean over all sequences) ─
# Layout : 3 rows (lead-0/1/2) × 2 cols (PSD power | PSD score)
# Colour : sub-region (harmonised palette).  Width : resolution (thick x2 / thin x50)
# Style  : solid = Reference, dashed = ML model.  Markers : per-region (black edge).
# xp     : experiment plotted (reads bench_psd[xp]). Change `xp` below to switch.
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

_REGIONS = ['Global', 'Barents Sea', 'Kara Sea', 'Greenland / Fram']
_REGION_LABEL = {'Global': 'Whole Arctic', 'Barents Sea': 'Barents Sea',
                 'Kara Sea': 'Kara Sea', 'Greenland / Fram': 'Greenland / Fram'}
_COLORS  = {'Global': '#333333', **REGION_COLORS}      # harmonised, single source
_MARKERS = {'Global': 'o', 'Barents Sea': 's', 'Kara Sea': '^', 'Greenland / Fram': 'D'}
_RES_LW  = {'x50': 1.0, 'x10': 2.0, 'x2': 3.2}         # thin = x50 … thick = x2
_NYQ_COLOR = {'x50': '#2166AC', 'x10': '#D6604D', 'x2': '#1A9850'}
SCORE_THRESH = 0.5
_FS = 11
xp = 'UNet-UOAI-res2'     # <-- experiment plotted in THIS cell


def _cross_05(wl, score):
    """First wavelength where the score crosses 0.5 (one value per curve)."""
    m = np.isfinite(wl) & np.isfinite(score)
    wl, score = wl[m], score[m]
    for i in range(len(score) - 1):
        if (score[i] - SCORE_THRESH) * (score[i + 1] - SCORE_THRESH) < 0:
            t = (SCORE_THRESH - score[i]) / (score[i + 1] - score[i])
            return float(np.exp(np.log(wl[i]) + t * (np.log(wl[i + 1]) - np.log(wl[i]))))
    return None


def _mk(ax, x, y, *, col, mrk, lw, ls, log=False):
    """Line with region-coloured markers (black edge), markers thinned out."""
    plot = ax.loglog if log else ax.semilogx
    me = max(1, len(x) // 7)
    plot(x, y, color=col, lw=lw, ls=ls, marker=mrk, markevery=me,
         ms=6, mfc=col, mec='black', mew=0.8, zorder=3, label='_nolegend_')


# Share x (wavelength) across all; share y per column (PSD col / Score col)
fig, axes = plt.subplots(3, 2, figsize=(15, 6.2 * 3),
                         sharex='all', sharey='col', constrained_layout=False)
fig.subplots_adjust(hspace=0.55, wspace=0.22, bottom=0.13, top=0.88)

for lt in range(3):
    ax_psd, ax_score = axes[lt, 0], axes[lt, 1]
    cut_entries = []                      # one (wavelength, colour) per curve

    for rname in _REGIONS:
        col, mrk = _COLORS[rname], _MARKERS[rname]
        for key in ['x50', 'x10', 'x2']:
            psd = bench_psd.get(xp, {}).get(key, {}).get(rname, {}).get(lt)
            if psd is None:
                continue
            wn, p_ref, p_study, p_diff = psd
            wl = 1.0 / np.where(wn > 0, wn, np.nan)
            score = 1.0 - p_diff / p_ref
            lw = _RES_LW[key]
            _mk(ax_psd, wl, p_ref,   col=col, mrk=mrk, lw=lw, ls='-',  log=True)
            _mk(ax_psd, wl, p_study, col=col, mrk=mrk, lw=lw, ls='--', log=True)
            _mk(ax_score, wl, score, col=col, mrk=mrk, lw=lw, ls='-')
            rs_km = _cross_05(wl, score)          # first crossing only -> one per curve
            if rs_km is not None:
                ax_score.axvline(rs_km, color=col, lw=0.9, ls=':', alpha=0.55,
                                 zorder=2, label='_nolegend_')
                cut_entries.append((rs_km, col))

    for key in ['x50', 'x10', 'x2']:
        nyq = 2.0 * cristal_data[key]['resolution_km']
        for ax in (ax_psd, ax_score):
            ax.axvline(nyq, color=_NYQ_COLOR[key], lw=1.1, ls='-.', alpha=0.55,
                       zorder=2, label='_nolegend_')
    ax_score.axhline(SCORE_THRESH, color='black', lw=1.6, ls='--', alpha=0.75,
                     zorder=2, label='_nolegend_')

    for ax in (ax_psd, ax_score):
        ax.grid(True, which='both', alpha=0.18, ls='--')
        ax.tick_params(labelsize=_FS - 1)
    ax_psd.set_ylabel('PSD  (km)', fontsize=_FS)
    ax_score.set_ylabel('PSD Score  [1 − err/ref]', fontsize=_FS)

    # 0.5-cut wavelengths: on the TOP edge of the score box, 45°, small font,
    # 2-level stagger to avoid overlap; titles are padded up to clear them.
    trans = ax_score.get_xaxis_transform()
    for j, (rs_km, col) in enumerate(sorted(cut_entries)):
        ax_score.annotate(f'{rs_km:.0f}', xy=(rs_km, 1.0), xytext=(rs_km, 1.02 + 0.07 * (j % 2)),
                          xycoords=trans, textcoords=trans, rotation=45,
                          ha='left', va='bottom', fontsize=8.5, color=col,
                          fontweight='bold', annotation_clip=False)

    ax_psd.set_title(f'Lead-{lt} — PSD', fontsize=_FS, fontweight='bold', pad=30)
    ax_score.set_title(f'Lead-{lt} — Score', fontsize=_FS, fontweight='bold', pad=30)

# x shared -> invert once; ylim of the Score column set once (shared)
axes[0, 0].invert_xaxis()
axes[0, 1].set_ylim([-0.18, 1.05])
for ax in (axes[2, 0], axes[2, 1]):
    ax.set_xlabel('Wavelength (km)', fontsize=_FS)

# ── Legend on TWO lines: regions (row 1) then styles (row 2) ──────────────────
_region_legend = [
    mlines.Line2D([], [], color=_COLORS[r], ls='-', lw=2.4, marker=_MARKERS[r],
                  ms=7, mfc=_COLORS[r], mec='black', mew=0.8, label=_REGION_LABEL[r])
    for r in _REGIONS
]
_style_legend = [
    mlines.Line2D([], [], color='#555555', ls='-',  lw=2.4, label='Reference'),
    mlines.Line2D([], [], color='#555555', ls='--', lw=2.4, label='ML model'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=3.2, label='x2 (1 km)'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=2.0, label='x10 (5 km)'),
    mlines.Line2D([], [], color='#555555', ls='-',  lw=1.0, label='x50 (25 km)'),
    mlines.Line2D([], [], color='black',   ls='--', lw=1.5, alpha=0.8, label='Score = 0.5'),
]
fig.legend(handles=_region_legend, loc='lower center', bbox_to_anchor=(0.5, 0.075),
           ncols=len(_region_legend), frameon=True, fontsize=_FS - 0.5,
           handlelength=2.2, columnspacing=1.4)
fig.legend(handles=_style_legend, loc='lower center', bbox_to_anchor=(0.5, 0.028),
           ncols=len(_style_legend), frameon=True, fontsize=_FS - 0.5,
           handlelength=2.2, columnspacing=1.4)

fig.suptitle(
    f'{xp} — Forecast PSD score (mean over {len(SEQUENCES)} sequences)\n'
    '(solid = reference,  dashed = ML model,  thick = x2 / thin = x50)',
    fontsize=_FS + 1, fontweight='bold', y=0.985)
plt.show()


## 4. Multi-Method Benchmark — Score Summary Table

Compare **4 methods** at common resolutions using **shared spatial segments** (fair comparison).

| Method | Resolutions |
|--------|------------|
| UNet-Unrolling | x50 + x10 |
| UNet-Unrolling-res2 | x10 only |
| UNet-UOAI | x50 + x10 |
| UNet-UOAI-res2 | x10 only |

**Scores**:
- `nRMSE` = RMSE / σ(reference)  — global and per sub-region (Barents Sea, Kara Sea, Greenland/Fram)
- `resolved_km` = wavelength at which PSD score = 0.5 (spectral score threshold)

> **Same segments** are used for all methods at a given resolution (generated once from the method with the longest analysis period).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MULTI-METHOD BENCHMARK TABLE — metrics averaged over the 25 sequences, by lead time
#   μ {region} (methods)     = 1 − RMSE(pred, Ground Truth)/σ(GT)   over the field
#   μ {region} (Persistence) = 1 − RMSE(CIMR obs @ init day, GT)/σ(GT)
#                              at CIMR-obs pixels  ('persistence on the obs')
#   Resolved {region} = wavelength @ PSD score 0.5 on the pooled PSD (per region)
# NB: Persistence is obs-masked (CIMR SIC pixels) while the methods are
#     full-field — the baseline is indicative, not pixel-for-pixel comparable.
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, xarray as xr
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)

REGIONS_TBL = list(FC_REGIONS.keys())
SCORE_COLS  = [f'μ {r}' for r in REGIONS_TBL] + [f'Resolved {r}' for r in REGIONS_TBL]

def _obs_to_pred(obs_sic, obs_xc, obs_yc, xc_p, yc_p):
    dxp = xc_p[1]-xc_p[0]; dyp = yc_p[1]-yc_p[0]
    iy, ix = np.where(np.isfinite(obs_sic))
    vals = obs_sic[iy, ix]
    ixp = np.clip(np.round((obs_xc[ix]-xc_p[0])/dxp).astype(int), 0, len(xc_p)-1)
    iyp = np.clip(np.round((obs_yc[iy]-yc_p[0])/dyp).astype(int), 0, len(yc_p)-1)
    return vals, ixp, iyp

# ── Persistence baseline ON THE OBS (CIMR at the last analysis day) ─────────
pers_mu = {}
for res in ['x10', 'x50', 'x2']:
    ref_method = next(m for m in METHODS if res in METHODS[m]['res'])
    acc = {(r, lt): [] for r in REGIONS_TBL for lt in range(N_FORECAST)}
    for idx, *_ in SEQUENCES:
        fp = method_file(ref_method, idx, res)
        if fp is None: continue
        with xr.open_dataset(fp) as ds:
            times = [str(t)[:10] for t in ds.time.values]
            gt_fc = ds[GT_VAR[res]].isel(time=slice(-N_FORECAST, None)).values
            xc_p, yc_p = ds.xc.values, ds.yc.values
            lat_p = ds['lat'].values; lon_p = (ds['lon'].values + 180) % 360 - 180
        init_date = times[-(N_FORECAST+1)]                  # last analysis day
        try:
            with xr.open_dataset(f'{CIMR_DIR}/CIMR5km_{init_date}.nc') as o:
                obs_sic = o['SIC'].squeeze().values; obs_xc = o.xc.values; obs_yc = o.yc.values
        except FileNotFoundError:
            continue
        vals, ixp, iyp = _obs_to_pred(obs_sic, obs_xc, obs_yc, xc_p, yc_p)
        if len(vals) == 0: continue
        la = lat_p[iyp, ixp]; lo = lon_p[iyp, ixp]
        for lt in range(N_FORECAST):
            gv_all = gt_fc[lt][iyp, ixp]
            for rname, ext in FC_REGIONS.items():
                rm = np.ones(len(vals), bool) if ext is None else \
                     ((la>=ext[2])&(la<=ext[3])&(lo>=ext[0])&(lo<=ext[1]))
                ok = rm & np.isfinite(gv_all) & np.isfinite(vals)
                if ok.sum() < 10: continue
                g = gv_all[ok]; sd = np.std(g)
                if sd < 1e-9: continue
                acc[(rname, lt)].append(1.0 - float(np.sqrt(np.mean((vals[ok]-g)**2))/sd))
    pers_mu[res] = {r: {lt: (float(np.mean(acc[(r, lt)])) if acc[(r, lt)] else np.nan)
                        for lt in range(N_FORECAST)} for r in REGIONS_TBL}

# ── Build table: Persistence (obs) baseline + the methods ─────────────────────
rows = []
for res in ['x10', 'x50', 'x2']:
    for lt in range(N_FORECAST):
        row = {'Res': res, 'Lead': f'Lead-{lt}', 'Method': 'Persistence (obs)'}
        for rname in REGIONS_TBL:
            row[f'μ {rname}'] = pers_mu[res][rname][lt]
            row[f'Resolved {rname}'] = np.nan
        rows.append(row)
    for method in [m for m in METHODS if res in METHODS[m]['res']]:
        for lt in range(N_FORECAST):
            row = {'Res': res, 'Lead': f'Lead-{lt}', 'Method': method}
            for rname in REGIONS_TBL:
                row[f'μ {rname}'] = bench_mu.get(method, {}).get(res, {}).get(rname, {}).get(lt, np.nan)
                row[f'Resolved {rname}'] = bench_resolved.get(method, {}).get(res, {}).get(rname, {}).get(lt, np.nan)
            rows.append(row)

_order = {'Persistence (obs)': 0}
df_bench = pd.DataFrame(rows)
df_bench['_mo'] = df_bench['Method'].map(lambda m: _order.get(m, 1))
df_bench = (df_bench.sort_values(['Res', 'Lead', '_mo', 'Method'])
                    .drop(columns='_mo').set_index(['Res', 'Lead', 'Method'])[SCORE_COLS])
df_bench = df_bench.round({**{c: 3 for c in SCORE_COLS if c.startswith('μ')},
                           **{c: 0 for c in SCORE_COLS if c.startswith('Resolved')}})

print("=" * 90)
print(f"SIC BENCHMARK — metrics averaged over {len(SEQUENCES)} sequences, by lead time")
print("  μ methods     = 1 − RMSE(pred, Ground Truth)/σ(GT)        (full field)")
print("  μ Persistence = 1 − RMSE(CIMR obs @ init day, GT)/σ(GT)  (at obs pixels)")
print("=" * 90)
try:
    display(df_bench)
except NameError:
    pass
print(df_bench.to_string())


### LaTeX export — table ready to copy into a .tex file

Rebuilt directly from `df_bench` (the DataFrame computed above), so it stays in sync with the notebook's numbers on every re-run.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LaTeX EXPORT — same benchmark table as above, recomputed from df_bench so it
# always reflects the current run (no hard-coded numbers).
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np

REGION_LABEL = {r: r.replace(' Sea', '').replace(' / ', '/') for r in REGIONS_TBL}
MU_COLS       = [f'μ {r}' for r in REGIONS_TBL]
RESOLVED_COLS = [f'Resolved {r}' for r in REGIONS_TBL]

def _fmt_mu(v):
    return '--' if pd.isna(v) else f'{v:.3f}'

def _fmt_resolved(v):
    return '--' if pd.isna(v) else f'{v:.0f}'

lines = []
lines.append(r'\begin{table*}[t]')
lines.append(r'\centering')
lines.append(r'\caption{Mean correlation skill ($\mu$) and resolved scales for each method, resolution and forecast lead --- Sea Ice Concentration (SIC), averaged over ' + str(len(SEQUENCES)) + ' sequences.}')
lines.append(r'\label{tab:sic_skill_resolved}')
lines.append(r'\small')
lines.append(r'\setlength{\tabcolsep}{4pt}')
lines.append(r'\begin{tabular}{ll l ' + 'c' * len(REGIONS_TBL) + ' ' + 'c' * len(REGIONS_TBL) + '}')
lines.append(r'\toprule')
lines.append(r'\multirow{2}{*}{Res.} &')
lines.append(r'\multirow{2}{*}{Lead} &')
lines.append(r'\multirow{2}{*}{Method} &')
lines.append(r'\multicolumn{' + str(len(REGIONS_TBL)) + r'}{c}{Mean correlation skill $\mu$} &')
lines.append(r'\multicolumn{' + str(len(REGIONS_TBL)) + r'}{c}{Resolved scale} \\')
lines.append(r'\cmidrule(lr){4-' + str(3 + len(REGIONS_TBL)) + '}')
lines.append(r'\cmidrule(lr){' + str(4 + len(REGIONS_TBL)) + '-' + str(3 + 2 * len(REGIONS_TBL)) + '}')
lines.append(' & & &\n' + ' & '.join(REGION_LABEL[r] for r in REGIONS_TBL) + ' &\n'
             + ' & '.join(REGION_LABEL[r] for r in REGIONS_TBL) + r' \\')
lines.append(r'\midrule')

ncols = 3 + 2 * len(REGIONS_TBL)
res_list = list(dict.fromkeys(df_bench.index.get_level_values('Res')))
for ri, res in enumerate(res_list):
    df_res = df_bench.xs(res, level='Res', drop_level=False)
    lead_list = list(dict.fromkeys(df_res.index.get_level_values('Lead')))
    n_rows_res = len(df_res)
    lines.append('')
    res_label = res.replace('x', r'\times ')
    lines.append(r'\multirow{' + str(n_rows_res) + '}{*}{$' + res_label + '$}')
    for li, lead in enumerate(lead_list):
        df_lead = df_res.xs(lead, level='Lead', drop_level=False)
        methods = list(df_lead.index.get_level_values('Method'))
        n_rows_lead = len(df_lead)
        lead_num = lead.split('-')[-1]
        for mi, method in enumerate(methods):
            row = df_lead.loc[(res, lead, method)]
            mu_vals  = ' & '.join(_fmt_mu(row[c]) for c in MU_COLS)
            res_vals = ' & '.join(_fmt_resolved(row[c]) for c in RESOLVED_COLS)
            if mi == 0:
                prefix = '& \\multirow{' + str(n_rows_lead) + '}{*}{' + str(lead_num) + '}\n& '
            else:
                prefix = '& & '
            lines.append(f'{prefix}{method:<22} & {mu_vals} & {res_vals} \\\\')
        if li < len(lead_list) - 1:
            lines.append(r'\cmidrule(lr){2-' + str(ncols) + '}')
    if ri < len(res_list) - 1:
        lines.append(r'\midrule')

lines.append('')
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table*}')

latex_table = '\n'.join(lines)
print(latex_table)


## 5. Per-sequence summary — when does multires beat res2?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PER-SEQUENCE SUMMARY — when does multires beat the dedicated res2 model?
# RMSE (mean over the 3 lead times) of multires-x10 vs res2-x10, per sequence.
# ══════════════════════════════════════════════════════════════════════════════
import xarray as xr, numpy as np, pandas as pd, datetime

_TS, _TE = datetime.date(2022, 5, 1), datetime.date(2022, 12, 31)   # training window

def _rmse_mean(fp):
    ds = xr.open_dataset(fp); p = ds['pred_SIC'].values; g = ds[GT_VAR['x2']].values; ds.close()
    na = p.shape[0] - N_FORECAST
    v = []
    for lt in range(N_FORECAST):
        a, b = p[na + lt], g[na + lt]; m = np.isfinite(a) & np.isfinite(b)
        v.append(np.sqrt(np.mean((a[m] - b[m]) ** 2)))
    return float(np.mean(v))

PAIRS = [('UOAI', 'UNet-UOAI', 'UNet-UOAI-res2'),
         ('Unroll', 'UNet-Unrolling', 'UNet-Unrolling-res2')]
rows = []
for idx, start, end in SEQUENCES:
    grp = 'in-dist' if _TS <= datetime.date.fromisoformat(end) <= _TE else 'OOD'
    row = {'seq': idx, 'start': start, 'group': grp}
    for tag, multi, res2 in PAIRS:
        fm, fr = method_file(multi, idx, 'x2'), method_file(res2, idx, 'x2')
        rm = _rmse_mean(fm) if fm else np.nan
        rr = _rmse_mean(fr) if fr else np.nan
        row[f'{tag} multi']  = round(rm, 3)
        row[f'{tag} res2']   = round(rr, 3)
        row[f'{tag} Δ(r-m)'] = round(rr - rm, 3)
        row[f'{tag} win']    = ('multi' if rm < rr else 'res2') if np.isfinite(rm) and np.isfinite(rr) else '—'
    rows.append(row)

df_seq = pd.DataFrame(rows).set_index('seq')
print("=" * 100)
print("PER-SEQUENCE RMSE (mean over lead times) — multires (corrected) vs res2")
print("  Δ(r-m) > 0  ⇒  multires better")
print("=" * 100)
try:
    display(df_seq)
except NameError:
    pass
print(df_seq.to_string())

# Summary: how often / where multires wins
for tag, multi, res2 in PAIRS:
    w = (df_seq[f'{tag} win'] == 'multi')
    print(f"\n{multi}: multires wins {int(w.sum())}/{len(df_seq)} seq  "
          f"| in-dist {int((w & (df_seq.group=='in-dist')).sum())}/{int((df_seq.group=='in-dist').sum())}  "
          f"OOD {int((w & (df_seq.group=='OOD')).sum())}/{int((df_seq.group=='OOD').sum())}  "
          f"| mean multi={df_seq[f'{tag} multi'].mean():.3f}  res2={df_seq[f'{tag} res2'].mean():.3f}")
